# Phase 1: ViewState Export/Import (Rust Backend)

This notebook validates export/import behavior with rebased identity/state fields on the Rust daemon.

In [1]:
from __future__ import annotations

import os
import shutil
import sys
import tempfile
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'tests').exists():
            return candidate
    raise RuntimeError('could not locate repository root from notebook cwd')


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
if str(REPO_ROOT / 'tests') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'tests'))

from lucida.client import LucidaClient
from conftest import create_sample_omezarr
from rust_daemon import start_rust_daemon


In [2]:
tmp_dir = Path(tempfile.mkdtemp(prefix='lucida-phase1-transfer-rust-'))
dataset_uri = create_sample_omezarr(str(tmp_dir / 'transfer.zarr'))
daemon = start_rust_daemon(repo_root=REPO_ROOT, env=dict(os.environ))
client = LucidaClient(base_url=daemon.base_url)
print({'dataset_uri': dataset_uri, 'base_url': daemon.base_url})


{'dataset_uri': '/var/folders/hs/qw7ws1q52153c4c639t_p3600000gn/T/lucida-phase1-transfer-rust-2fbonjl2/transfer.zarr', 'base_url': 'http://127.0.0.1:50706'}


In [3]:
source_session = client.create_session()
opened = client.open_dataset(dataset_uri, session_id=source_session.session_id)
created = client.create_view(dataset_id=opened.dataset_summary.dataset_id, session_id=source_session.session_id)

source_view = client.set_dim(
    view_id=created.view_state.view_id,
    axis='z',
    index=2,
    session_id=source_session.session_id,
).view_state

exported = client.export_viewstate(view_id=source_view.view_id, session_id=source_session.session_id)
assert exported.export_id.startswith('exp_')
assert exported.source_view_id == source_view.view_id
assert exported.view_state.state_hash == source_view.state_hash

{
    'source_session_id': source_session.session_id,
    'source_view_id': source_view.view_id,
    'source_state_version': source_view.state_version,
}


{'source_session_id': 'session_c6b85f06a08b41ef',
 'source_view_id': 'view_96deb72507344ab6',
 'source_state_version': 1}

In [4]:
target_session = client.create_session()
imported = client.import_viewstate(view_state=exported.view_state.model_dump(mode='json'), session_id=target_session.session_id)

assert imported.import_id.startswith('imp_')
assert imported.imported_from_view_id == source_view.view_id
assert imported.view_state.view_id != source_view.view_id
assert imported.view_state.session_id == target_session.session_id
assert imported.view_state.state_version == 0
assert bool(imported.view_state.state_hash)
assert imported.view_state.state_hash != source_view.state_hash

z_selector = next(item for item in imported.selectors_applied if item.axis == 'z')
assert z_selector.kind == 'index'
assert z_selector.index == 2

fetched = client.get_view(view_id=imported.view_state.view_id, session_id=target_session.session_id)
assert fetched.view_state.view_id == imported.view_state.view_id
assert fetched.view_state.session_id == target_session.session_id

{
    'target_session_id': target_session.session_id,
    'imported_view_id': imported.view_state.view_id,
    'imported_state_hash': imported.view_state.state_hash,
}


{'target_session_id': 'session_eb12f0d1bce84ab9',
 'imported_view_id': 'view_a76512f180004310',
 'imported_state_hash': 'd5ba1dc2dc106b6322ea20e86fea3f0ea9ab712adb690b35f61cf41d2291efb5'}

In [5]:
if 'client' in globals():
    client.close()
if 'daemon' in globals():
    daemon.stop()
if 'tmp_dir' in globals():
    shutil.rmtree(tmp_dir, ignore_errors=True)
